# PTB-XL 543-feature pipeline — Revision Experiments — **v2.7** (8g-3 family mapping fix)

**আপনার কাজ:** Drive থেকে Colab-এ open → **Runtime → Run all** (~25–30 মিনিট)।

**v2.7-এ কী বদলাল:** 8g-3 এর family mapping এখন explicit, Table 2 এর সাথে hুবহু মেলে (Morphology 182 / Intervals 168 / Complexity 145 / fQRS 27 / Global 15 / HRV 6)। mapping না মিললে cell নিজেই থেমে যাবে। শুধু 8g-3 আবার চলবে, 8g-1 ও 8g-2 আগের checkpoint থেকে নেওয়া হবে।

শেষ cell-এর COPY-PASTE SUMMARY (বা `/content/analysis_results.json`) আমাকে পাঠাবেন।


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass
import importlib, sys, subprocess
for pkg in ("shap", "lightgbm", "scipy"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        try:
            get_ipython().run_line_magic("pip", "install -q " + pkg)
        except NameError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import sys
print("Python", sys.version.split()[0], "| ready")


In [ ]:
import os, re, ast, json, time, warnings
import numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, confusion_matrix
from scipy.stats import spearmanr
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

print("NOTEBOOK VERSION: 2.5 (§8b KeyError fix + §8d no-HRV ablation + §8e per-fold SDs)")

# ================ CONFIG (auto-detection কাজ না করলেই এটা edit করুন) ================
DATA_DIR = "/content/drive/MyDrive/PTB-XL Dataset Analysis"
FEATURES = None   # None = auto-detect। ভুল file picked হলে এখানে exact path দিন
ML_FILE  = None   # None = auto-detect (multi-label 0/1 matrix)
PERMS    = 3      # label permutations (1 = দ্রুত)
N_JOBS   = 2
# =====================================================================================

CLASS_ORDER = ["NORM", "MI", "STTC", "CD", "HYP"]
NON_FEATURE_COLS = {"filename", "record_id", "patient_id", "subject_id", "strat_fold",
                    "label", "Disease_Name", "Disease_Label_Encoded",
                    "Disease_NORM", "Disease_MI", "Disease_STTC", "Disease_CD", "Disease_HYP",
                    "ECG_ID", "ecg_id"}
PARAMS = dict(objective="multiclass", num_class=5, learning_rate=0.05, n_estimators=800,
              num_leaves=63, min_child_samples=40, subsample=0.9, subsample_freq=1,
              colsample_bytree=0.7, reg_lambda=5.0, random_state=42, n_jobs=N_JOBS, verbose=-1)
R = {}

if not os.path.isdir(DATA_DIR):
    raise SystemExit("DATA_DIR not found: " + DATA_DIR +
                     "\n(Colab-এ Drive mounted আছে কিনা দেখুন, আর folder name-এর spelling ঠিক আছে কিনা)")

all_files = []
for root, _, fs in os.walk(DATA_DIR):
    if os.path.basename(root) == "_analysis_checkpoints":
        continue
    for f in fs:
        if f.startswith("."):
            continue
        p = os.path.join(root, f)
        if os.path.isfile(p):
            all_files.append((p, os.path.getsize(p)))
print("=== files in", DATA_DIR, "===")
for p, s in sorted(all_files):
    print(f"  {os.path.relpath(p, DATA_DIR)}   {s/1024/1024:.1f} MB")

for p, s in all_files:
    if p.lower().endswith(".json") and s < 200_000:
        print(f"\n=== {os.path.basename(p)} (audit file) ===")
        print(open(p).read()[:1200])

def find_file(*pats):
    for p, s in all_files:
        n = os.path.basename(p).lower()
        if any(x in n for x in pats):
            return p
    return None

PTBXL_DB = find_file("ptbxl_database")
print("\nPTBXL_DB:", os.path.basename(PTBXL_DB) if PTBXL_DB else "NOT FOUND")

def detect_columns(d):
    cols = list(d.columns)
    def find(*pats):
        for pat in pats:
            for c in cols:
                if pat.lower() in str(c).lower():
                    return c
        return None
    fold = find("strat_fold", "stratfold", "_fold")
    if fold is None:
        for c in cols:
            u = np.unique(d[c].dropna())
            if len(u) == 10 and set(u.tolist()).issubset(set(range(1, 11))):
                fold = c
                break
    patient = find("patient", "subject")
    idc = find("filename", "record_id", "ecg_id", "ecgid", "record id")
    lab = find("label", "diagnosis", "disease", "primary")
    return fold, patient, idc, lab

def n_features(d, fold, patient, lab, idc):
    excl = {fold, patient, lab, idc}
    return len([c for c in d.select_dtypes("number").columns
                if c not in excl and c not in NON_FEATURE_COLS and not str(c).startswith("Disease_")])

cand = []
for p, s in all_files:
    n = os.path.basename(p).lower()
    if n.endswith((".csv", ".parquet", ".pq")) and s > 300_000 \
       and "fqr" not in n and "database" not in n and "metadata" not in n and "scp_" not in n:
        cand.append(p)
def prio(p):
    n = os.path.basename(p).lower()
    sc = 0
    if "model_ready" in n and "stratfold" in n: sc -= 4
    elif "stratfold" in n: sc -= 3
    if "final" in n: sc -= 2
    if "master" in n: sc -= 1
    return sc
cand.sort(key=prio)
print("\n=== candidate feature files (priority order) ===")
for p in cand:
    print("  ", os.path.relpath(p, DATA_DIR))

valid = []
for p in cand:
    name = os.path.basename(p)
    print(f"\n--- {name} ---")
    try:
        d = pd.read_parquet(p) if p.endswith((".parquet", ".pq")) else pd.read_csv(p)
    except Exception as e:
        print("  load failed:", type(e).__name__, str(e)[:120])
        continue
    fold, patient, idc, lab = detect_columns(d)
    probs = [f"no {nm} column" for nm, c in (("fold", fold), ("patient", patient), ("label", lab)) if c is None]
    if not probs:
        fu = set(np.unique(d[fold].dropna()))
        if not (fu.issubset(set(range(1, 11))) or fu.issubset(set(range(10)))):
            probs.append(f"fold values {sorted(fu)[:12]}")
        lv = set(d[lab].astype(str).unique())
        if not (lv.issubset(set(CLASS_ORDER)) or lv.issubset({str(i) for i in range(5)})):
            probs.append(f"label values {sorted(lv)[:8]}")
        dup = d.groupby(patient)[fold].nunique()
        if (dup > 1).any():
            probs.append(f"{int((dup > 1).sum())} patients in >1 fold")
    nf = n_features(d, fold, patient, lab, idc)
    print(f"  rows={len(d)}  feature_cols={nf}  fold={fold}  patient={patient}  id={idc}  label={lab}")
    if lab is not None:
        print("  label counts:", d[lab].value_counts().to_dict())
    print("  problems:", probs or "none")
    if not probs and nf >= 400:
        valid.append([p, d, fold, patient, idc, lab, nf])
        if len(d) == 20310:
            print("  >>> exact cohort size 20,310 — stopping the scan")
            break
    del d

if FEATURES:
    print("\n>>> FEATURES set manually:", FEATURES)
    d = pd.read_parquet(FEATURES) if FEATURES.endswith((".parquet", ".pq")) else pd.read_csv(FEATURES)
    fold, patient, idc, lab = detect_columns(d)
    if not (fold and patient and lab):
        raise SystemExit(f"manual FEATURES given but columns not detected: fold={fold} patient={patient} label={lab} — set them manually below")
    valid = [[FEATURES, d, fold, patient, idc, lab, n_features(d, fold, patient, lab, idc)]]
if not valid:
    raise SystemExit("NO candidate file passed validation. Set FEATURES manually in CONFIG and re-run this cell. "
                     "Send the report above if you are unsure which file to pick.")
valid.sort(key=lambda v: (0 if len(v[1]) == 20310 else 1, -v[6]))
FEATURES, df, FOLD_COL, PATIENT_COL, ID_COL, LABEL_COL, nf = valid[0]
for v in valid[1:]:
    del v[1]
print(f"\n>>> CHOSEN feature file: {os.path.basename(FEATURES)}   (rows={len(df)}, feature cols={nf})")
if len(df) != 20310:
    print(">>> NOTE: rows != 20,310 — verify this is the final cohort file!")

if ID_COL is None:
    raise SystemExit("STOP: record-id column not found (looked for filename / record_id / ecg_id / ecgid). "
                     "Do NOT run the sections — send me the column list printed above.")
if ID_COL == LABEL_COL:
    raise SystemExit("STOP: id detection picked the label column — send me the detection report above.")
print("id column:", ID_COL, "| dtype:", df[ID_COL].dtype, "| sample:", df[ID_COL].head(3).tolist(),
      "| unique:", int(df[ID_COL].nunique()), "/", len(df))
if df[ID_COL].nunique() != len(df):
    print(">>> WARNING: id column is not unique — id-based joins (Sections 4, 7) may be wrong")

fvals = set(np.unique(df[FOLD_COL]))
if fvals.issubset(set(range(10))):
    df[FOLD_COL] = df[FOLD_COL] + 1
    print(">>> fold values were 0-9, shifted to 1-10")

def _align_numeric_labels(y_raw):
    """Match each numeric label code's count to the canonical PTB-XL 5-class sizes
    (unique counts -> unambiguous) so codes align to [NORM,MI,STTC,CD,HYP].
    Self-verifying: any mismatch -> warn + assume order, never silent mislabel."""
    CANON = {"NORM": 9046, "MI": 3890, "STTC": 3107, "CD": 3044, "HYP": 1223}
    count_to_class = {v: k for k, v in CANON.items()}
    vc = pd.Series(y_raw).value_counts().to_dict()
    if len(vc) != 5:
        print(">>> numeric label column has", len(vc), "unique codes (expected 5); assuming order", CLASS_ORDER, "- VERIFY manually!")
        return y_raw
    remap = {}
    for code in sorted(vc):
        if vc[code] in count_to_class:
            remap[code] = CLASS_ORDER.index(count_to_class[vc[code]])
        else:
            print(">>> WARNING: a label code's count matches no canonical PTB-XL size; assuming order", CLASS_ORDER, "- VERIFY!")
            return y_raw
    if set(remap.values()) != set(range(5)):
        print(">>> WARNING: ambiguous count match; assuming order", CLASS_ORDER, "- VERIFY!")
        return y_raw
    if remap == {i: i for i in range(5)}:
        print(">>> numeric labels already in [NORM,MI,STTC,CD,HYP] order")
        return y_raw
    print(">>> REMAPPING numeric labels (file order != [NORM,MI,STTC,CD,HYP]):")
    for c in sorted(remap):
        print(f"      code {c} -> {CLASS_ORDER[remap[c]]}")
    return np.array([remap[c] for c in y_raw], dtype=int)

if df[LABEL_COL].dtype == object:
    y = df[LABEL_COL].map({c: i for i, c in enumerate(CLASS_ORDER)}).to_numpy()
else:
    y = _align_numeric_labels(df[LABEL_COL].to_numpy(dtype=int))

feature_cols = [c for c in df.select_dtypes("number").columns
                if c not in NON_FEATURE_COLS and c not in (FOLD_COL, PATIENT_COL, LABEL_COL, ID_COL)
                and not str(c).startswith("Disease_")]
X = df[feature_cols]
if ID_COL in feature_cols:
    raise SystemExit(f"STOP: {ID_COL} is inside the feature matrix — detection bug, send me the report above")
counts = pd.Series(y).value_counts().reindex(list(range(5)))
print("feature columns:", len(feature_cols), "  <- expect 543")
if len(feature_cols) != 543:
    print(">>> WARNING: feature count is not 543 — check that id / label / Disease_ columns are excluded before trusting any number")
print("class counts (expect 9046/3890/3107/3044/1223):", dict(zip(CLASS_ORDER, counts.astype(int).tolist())))
if counts.tolist() == [9046, 3890, 3107, 3044, 1223]:
    print(">>> LABEL ORDER CHECK: PASS (NORM 9046 / MI 3890 / STTC 3107 / CD 3044 / HYP 1223)")
else:
    print(">>> LABEL ORDER CHECK: MISMATCH — STOP and verify the label column (see REMAPPING line above) before running the sections!")
g = df.groupby(PATIENT_COL)[FOLD_COL].nunique()
assert (g == 1).all(), f"{(g > 1).sum()} patients in >1 fold"
print("patients:", df[PATIENT_COL].nunique(), "— OK")

if ML_FILE is None:
    for p, s in all_files:
        n = os.path.basename(p).lower()
        if not (n.endswith(".csv") and s < 20_000_000):
            continue
        try:
            d2 = pd.read_csv(p)
        except Exception:
            continue
        cc = {str(c).lower(): c for c in d2.columns}
        need = [cc.get(c) for c in ("norm", "mi", "sttc", "cd", "hyp")]
        if None in need:
            continue
        vals = set(np.unique(d2[need].to_numpy()))
        if not vals.issubset({0, 1, 0.0, 1.0}):
            continue
        if len(d2) < 5000:
            continue
        ML_FILE = p
        print(f"ML_FILE auto-picked: {os.path.basename(p)}  rows={len(d2)}  id_col={d2.columns[0]}")
        if len(d2) != 20192:
            print(">>> NOTE: ML matrix rows != 20,192 — confirm it is the CinC 10-fold set, else set ML_FILE = None")
        break
if ML_FILE:
    print("ML_FILE:", os.path.basename(ML_FILE))
elif PTBXL_DB:
    print("ML_FILE: none — Section 7 will derive the multi-label 0/1 ground truth from ptbxl_database.csv (CinC-2020 superclass rule)")
else:
    print("ML_FILE: none and no ptbxl_database.csv — Section 7 will be skipped")

MI_COL = None
for c in df.columns:
    if re.search(r"^mi[_-]?(superclass|ind|flag|code|label)", str(c), re.I):
        MI_COL = c
        break
print("MI_COL:", MI_COL or "(MI co-occurrence will use PTBXL_DB)")

# ---------------- CHECKPOINT / RESUME ----------------
# প্রতিটা section শেষ হলে এখানে save হয়। Runtime মারা গেলে আবার Run all দিলে
# done section গুলো auto-skip হবে — আরম্ভ হবে শেষ checkpoint থেকে।
import shutil as _shutil
CKPT_DIR = os.path.join(DATA_DIR, "_analysis_checkpoints")
try:
    os.makedirs(CKPT_DIR, exist_ok=True)
except Exception:
    CKPT_DIR = "/content"
RERUN = ["8g3"]  # v2.7: only the family ablation re-runs with the exact Table 2 mapping (~25 min). 8b–8e, 8g1, 8g2 are kept from the checkpoint
FORCE_RERUN = False    # True = পুরো checkpoint ignore করে শুরু থেকে
if FORCE_RERUN and os.path.exists(os.path.join(CKPT_DIR, "state.json")):
    os.remove(os.path.join(CKPT_DIR, "state.json"))
    print("[checkpoint] FORCE_RERUN=True — পুরনো checkpoint মুছে ফেলা হলো")
R, DONE = {}, set()
if os.path.exists(os.path.join(CKPT_DIR, "state.json")):
    _s = json.load(open(os.path.join(CKPT_DIR, "state.json")))
    R, DONE = _s["R"], set(_s["DONE"])
    _r = set(str(x) for x in RERUN)
    DONE -= _r
    print(f"[checkpoint] পুরনো run পাওয়া গেছে — done: {sorted(DONE)} (এই section গুলো skip হবে)")
    if _r:
        print(f"[checkpoint] RERUN={sorted(_r)} — এই section গুলো আবার হিসাব হবে")
else:
    print(f"[checkpoint] নতুন run — প্রতিটা section শেষে save হবে: {CKPT_DIR}")
def _done(sec):
    DONE.add(str(sec))
    for _f in [f for f in os.listdir(".")
                 if (f.startswith("probs_oof_") or f.startswith("figure")) and (f.endswith(".csv") or f.endswith(".png"))]:
        if os.path.exists(_f):
            _shutil.copy(_f, CKPT_DIR)
    json.dump({"R": R, "DONE": sorted(DONE), "ts": time.strftime("%Y-%m-%d %H:%M:%S")},
              open(os.path.join(CKPT_DIR, "state.json"), "w"), default=str)
    print(f"[CHECKPOINT] section {sec} done — saved — এখানে থামা/ভুলে যাওয়া safe")
print("\nLOAD + AUTO-DETECTION DONE")


In [ ]:
def class_sample_weights(y_train, K=5):
    """Eq. 13: w_c = N / (K * N_c), applied per sample (== LightGBM class_weight)."""
    N = len(y_train)
    counts = np.bincount(y_train, minlength=K)
    w = N / (K * np.maximum(counts, 1))
    return w[y_train]

def run_cv(df, X, y, feature_cols, exclude=(), tag=""):
    """Patient-level 10-fold: val fold 9 (or 8 when test fold k == 9), early-stop patience 50."""
    feats = [c for c in feature_cols if c not in set(exclude)]
    folds = df[FOLD_COL].to_numpy()
    n = len(df)
    probs_oof = np.zeros((n, 5))
    t0 = time.time()
    for k in range(1, 11):
        test = np.where(folds == k)[0]
        val = np.where(folds == (8 if k == 9 else 9))[0]
        train = np.setdiff1d(np.arange(n), np.union1d(test, val))
        m = lgb.LGBMClassifier(**PARAMS)
        m.fit(X.loc[train, feats].to_numpy(), y[train], sample_weight=class_sample_weights(y[train]),
              eval_set=[(X.loc[val, feats].to_numpy(), y[val])],
              eval_metric="multi_logloss",
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        probs_oof[test] = m.predict_proba(X.loc[test, feats].to_numpy())
        try:
            fa = roc_auc_score(y[test], probs_oof[test], multi_class='ovr', average='macro')
            fa_s = f"{fa:.4f}"
        except ValueError:
            fa_s = "n/a (class missing in fold)"
        print(f"  [{tag}] fold {k}: test AUC {fa_s}  (iter {m.best_iteration_})")
    print(f"  [{tag}] done in {(time.time()-t0)/60:.1f} min")
    return probs_oof

def metrics(y, probs):
    pred = probs.argmax(axis=1)
    out = dict(accuracy=float(accuracy_score(y, pred)),
               macro_f1=float(f1_score(y, pred, average="macro")),
               macro_auc=float(roc_auc_score(y, probs, multi_class="ovr", average="macro")),
               per_class_auc={c: float(roc_auc_score((y == i).astype(int), probs[:, i])) for i, c in enumerate(CLASS_ORDER)},
               per_class_f1={c: float(f1_score((y == i).astype(int), pred == i)) for i, c in enumerate(CLASS_ORDER)})
    cm = confusion_matrix(y, pred, labels=list(range(5)))
    out["error_breakdown"] = dict(total_errors=int(cm.sum() - np.trace(cm)),
                                  diseased_to_NORM=int(cm[1:5, 0].sum()),
                                  NORM_to_diseased=int(cm[0, 1:5].sum()),
                                  disease_to_disease=int(cm[1:5, 1:5].sum() - np.trace(cm[1:5, 1:5])),
                                  confusion_matrix=cm.tolist())
    return out

_BUILTIN_SC = {
    # official PTB-XL (v1.0.0-1.0.3) diagnostic statements -> superclass — 44 codes,
    # verified identical to the official scp_statements.csv (physionet ptb-xl/1.0.3)
    "NORM": "NORM",
    "IMI": "MI", "ASMI": "MI", "ILMI": "MI", "AMI": "MI", "ALMI": "MI", "INJAS": "MI",
    "LMI": "MI", "INJAL": "MI", "IPLMI": "MI", "IPMI": "MI", "INJIN": "MI", "INJLA": "MI",
    "PMI": "MI", "INJIL": "MI",
    "NDT": "STTC", "NST_": "STTC", "DIG": "STTC", "LNGQT": "STTC", "ISC_": "STTC",
    "ISCAL": "STTC", "ISCIN": "STTC", "ISCIL": "STTC", "ISCAS": "STTC", "ISCLA": "STTC",
    "ANEUR": "STTC", "EL": "STTC", "ISCAN": "STTC",
    "LAFB": "CD", "IRBBB": "CD", "1AVB": "CD", "IVCD": "CD", "CRBBB": "CD", "CLBBB": "CD",
    "LPFB": "CD", "WPW": "CD", "ILBBB": "CD", "3AVB": "CD", "2AVB": "CD",
    "LVH": "HYP", "LAO/LAE": "HYP", "RVH": "HYP", "RAO/RAE": "HYP", "SEHYP": "HYP",
}

def load_superclass_map():
    """{SCP code: superclass} over the 44 official diagnostic statements.
    Prefers scp_statements.csv (column 'superclass' or 'diagnostic_class'); else built-in
    (which is the official v1.0.3 map itself)."""
    cands = []
    if PTBXL_DB:
        cands.append(os.path.join(os.path.dirname(PTBXL_DB) or ".", "scp_statements.csv"))
    if "DATA_DIR" in globals() and os.path.isdir(DATA_DIR):
        cands.append(os.path.join(DATA_DIR, "scp_statements.csv"))
    for c in cands:
        if os.path.exists(c):
            try:
                s = pd.read_csv(c)
            except Exception:
                print("NOTE: scp_statements.csv unreadable — using built-in map (official 44 codes)")
                continue
            code_col = next((x for x in s.columns if str(x).lower() in ("code", "scp_code")), s.columns[0])
            sc_col = next((x for x in s.columns
                           if "superclass" in str(x).lower() or "diagnostic_class" in str(x).lower()), None)
            if sc_col is None:
                print("NOTE: scp_statements.csv found but no superclass/diagnostic_class column — using built-in map (official 44 codes)")
                continue
            m = {}
            for _, row in s.iterrows():
                if pd.notna(row[sc_col]) and str(row[sc_col]).strip():
                    m[str(row[code_col]).strip()] = str(row[sc_col]).strip().upper()
            if len(m) >= 40:
                from collections import Counter as _C
                print("superclass map: from", os.path.basename(c), f"({len(m)} codes | "
                      + ", ".join(f"{k} {v}" for k, v in sorted(_C(m.values()).items())) + ")")
                return m
            print("NOTE: scp_statements.csv parsed but only", len(m),
                  "diagnostic codes (expected 44) — using built-in map (official 44 codes)")
    print("superclass map: built-in official PTB-XL 44 diagnostic codes (scp_statements.csv not found)")
    return dict(_BUILTIN_SC)

def db_indexed():
    """PTBXL_DB indexed by its id column. Returns (db, scp_codes_column) or None."""
    if not PTBXL_DB:
        return None
    db = pd.read_csv(PTBXL_DB)
    idc = next((c for c in db.columns if str(c).lower() in
                {"filename", "record_id", "ecgid", "ecg_id", "file_name", "id"}), None)
    if idc is None:
        idc = db.columns[0]
        print("NOTE: database id column not obviously named; using first column:", idc)
    scpc = next((c for c in db.columns if "scp" in str(c).lower() and "code" in str(c).lower()), None)
    if scpc is None:
        non_id = [c for c in db.columns if c != idc]
        scpc = non_id[0] if non_id else None
        if scpc is not None:
            print("NOTE: scp_codes column not obviously named; using:", scpc)
    if scpc is None:
        print("WARNING: could not find id/scp_codes columns in", PTBXL_DB, "— columns:", list(db.columns))
        return None
    print("database: id column =", idc, "| scp column =", scpc)
    return db.set_index(idc), scpc

def _codes_to_flags(s, scmap):
    """scp_codes string -> [NORM, MI, STTC, CD, HYP] 0/1 flags (CinC-2020 rule: any statement of the superclass)."""
    try:
        codes = ast.literal_eval(s)
    except Exception:
        return [0, 0, 0, 0, 0]
    flags = [0, 0, 0, 0, 0]
    for c in codes:
        sc = scmap.get(str(c).strip())
        if sc in CLASS_ORDER:
            flags[CLASS_ORDER.index(sc)] = 1
    return flags

def mi_indicator(df):
    """0/1: record carries >= 1 MI superclass statement (from MI_COL or PTB-XL database)."""
    if MI_COL and MI_COL in df.columns:
        return df[MI_COL].fillna(0).astype(int).to_numpy()
    _db = db_indexed()
    if _db is None:
        print("WARNING: no MI_COL / PTBXL_DB available — MI co-occurrence panel will be skipped")
        return None
    dbx, scpc = _db
    scmap = load_superclass_map()
    matched = int(df[ID_COL].isin(dbx.index).sum())
    print(f"MI join: {matched}/{len(df)} feature-file ids found in the database")
    if matched < 0.95 * len(df):
        print("WARNING: id join failed — feature ids sample:", df[ID_COL].head(3).tolist(),
              "| database ids sample:", list(dbx.index[:3]))
    mi = np.array([_codes_to_flags(s, scmap)[1] for s in dbx[scpc].reindex(df[ID_COL])], dtype=int)
    if (mi == 0).all():
        print("WARNING: no record matched an MI statement — check the id columns above")
    return mi
print("HELPERS LOADED")

## ID alignment check (runs every time, ~10 sec)
৩ ঘণ্টার section গুলোর আগে: feature file-এর record id আর `ptbxl_database.csv`-এর id align করছে কিনা সেটা নিজেই দেখে নেয়।
**`ID ALIGNMENT: PASS` না দেখলে** Section 4/7-এর সংখ্যা trust করবেন না — এই block-টা আমাকে পাঠান।


In [ ]:
print("=== ID ALIGNMENT CHECK (runs every time) ===")
if PTBXL_DB:
    _chk = db_indexed()
    if _chk is not None:
        dbx, _scpc = _chk
        _f = df[ID_COL]
        _d = dbx.index
        print(f"feature-file id: {ID_COL}  dtype={_f.dtype}  sample={_f.head(3).tolist()}  min={_f.min()}  max={_f.max()}  unique={_f.nunique()}/{len(_f)}")
        print(f"database id:     {dbx.index.name}  dtype={_d.dtype}  sample={_d[:3].tolist()}  min={_d.min()}  max={_d.max()}  unique={_d.nunique()}/{len(_d)}")
        _m = int(_f.isin(_d).sum())
        print(f"matched: {_m} / {len(_f)}  ({_m / len(_f):.1%})")
        if _m < len(_f):
            print("ID ALIGNMENT: FAIL — the two files' id systems do not line up.")
            print("   Likely causes: (a) different id spaces, (b) 0-based vs 1-based numbering, (c) wrong id column detected.")
            print("   Send me this whole block. Sections 4 and 7 (database joins) will be unreliable until fixed.")
        else:
            # semantic cross-check: catches off-by-one numbering where raw ids still overlap
            _scmap = load_superclass_map()
            _flags = np.array([_codes_to_flags(s, _scmap) for s in dbx[_scpc].reindex(_f)], dtype=int)
            _agree = float((_flags[np.arange(len(_f)), y] == 1).mean())
            print(f"semantic cross-check (feature label found in db superclass set): {_agree:.1%}")
            if _agree < 0.9:
                print("ID ALIGNMENT: SUSPECT — raw ids match but labels don't line up (off-by-one numbering likely).")
                print("   Send me this whole block. Sections 4 and 7 will be unreliable.")
            else:
                print("ID ALIGNMENT: PASS")
    else:
        print("ID ALIGNMENT: skipped — database columns not parseable (see warnings above)")
else:
    print("ID ALIGNMENT: skipped — no ptbxl_database.csv found")
print("ID CHECK DONE")

## 1) Baseline 10-fold — must reproduce Table 3 (acc 0.762 / macro-F1 0.702 / macro-AUC 0.935)

In [ ]:
_SEC = "1"
if _SEC in DONE and _SEC not in RERUN:
    print(f"=== 1) already completed (checkpoint) — skipping")
else:
    print("=== 1) BASELINE ===")
    probs_oof, = [run_cv(df, X, y, feature_cols, tag="base")]
    m_base = metrics(y, probs_oof)
    pd.DataFrame(probs_oof, columns=CLASS_ORDER, index=df[ID_COL]).to_csv("probs_oof_weighted.csv", index_label=ID_COL)
    eb = m_base["error_breakdown"]
    print(f"pooled OOF: acc {m_base['accuracy']:.4f} | macroF1 {m_base['macro_f1']:.4f} | macroAUC {m_base['macro_auc']:.4f}")
    print(f"error breakdown: total {eb['total_errors']} | diseased->NORM {eb['diseased_to_NORM']} | "
          f"NORM->diseased {eb['NORM_to_diseased']} | disease->disease {eb['disease_to_disease']}")
    print("confusion matrix (rows/cols NORM,MI,STTC,CD,HYP):"); print(pd.DataFrame(eb["confusion_matrix"], index=CLASS_ORDER, columns=CLASS_ORDER))
    R["baseline"] = m_base
    print("SECTION 1 DONE")
    _done("1")


## 2) Label-permutation test — pooled macro-AUC must be ~0.50 (⚠ stop if > 0.55)

In [ ]:
_SEC = "2"
if _SEC in DONE and _SEC not in RERUN:
    print(f"=== 2) already completed (checkpoint) — skipping")
else:
    print(f"=== 2) LABEL PERMUTATION x{PERMS} ===")
    rng = np.random.default_rng(42)
    perm_aucs = []
    for r in range(PERMS):
        y_perm = rng.permutation(y)
        probs_p = run_cv(df, X, y_perm, feature_cols, tag=f"perm{r+1}")
        a = float(roc_auc_score(y_perm, probs_p, multi_class="ovr", average="macro"))
        perm_aucs.append(a)
        print(f"  perm {r+1}: pooled macro-AUC = {a:.4f}   (expect ~0.50)")
    R["permutation"] = dict(n=PERMS, aucs=[round(a, 4) for a in perm_aucs],
                            mean=round(float(np.mean(perm_aucs)), 4),
                            range=[round(float(np.min(perm_aucs)), 4), round(float(np.max(perm_aucs)), 4)])
    print("SECTION 2 DONE")
    _done("2")


## 3) Single-feature AUC scan + leave-out — (⚠ stop if Δ macro-AUC > 0.02)

In [ ]:
_SEC = "3"
if _SEC in DONE and _SEC not in RERUN:
    print(f"=== 3) already completed (checkpoint) — skipping")
else:
    if "baseline" not in R:
        raise SystemExit("Section 3 needs Section 1's result (R['baseline']) — Section 1 আগে complete করুন")
    m_base = R["baseline"]
    print("=== 3) SINGLE-FEATURE SCAN + LEAVE-OUT ===")
    Xs = X[feature_cols].fillna(X[feature_cols].median())
    tops = {}
    for i, c in enumerate(CLASS_ORDER):
        yb = (y == i).astype(int)
        aucs = np.array([0.5 if np.unique(Xs[f].to_numpy()).size < 2 else roc_auc_score(yb, Xs[f].to_numpy())
                         for f in feature_cols])
        j = int(np.argmax(aucs))
        tops[c] = (feature_cols[j], round(float(aucs[j]), 3))
        print(f"  top-1 for {c}: {feature_cols[j]} (AUC {aucs[j]:.3f})")
    # manuscript says: NORM 0.933 I_Q_Amplitude | MI 0.844 III_fQRS_Flag | STTC 0.899 V3_Q_R_Ratio |
    #                  CD 0.906 V5_Q_Amplitude | HYP 0.929 I_Q_Amplitude  <- compare!
    drop = sorted({f for f, _ in tops.values()})
    print("dropping:", drop)
    probs_lo = run_cv(df, X, y, feature_cols, exclude=drop, tag="leave-out")
    m_lo = metrics(y, probs_lo)
    print(f"macro-AUC: base {m_base['macro_auc']:.4f} -> leave-out {m_lo['macro_auc']:.4f}  (delta {m_base['macro_auc']-m_lo['macro_auc']:+.4f})")
    for c in CLASS_ORDER:
        print(f"  {c:5s} AUC {m_base['per_class_auc'][c]:.4f} -> {m_lo['per_class_auc'][c]:.4f}")
    R["leave_out"] = dict(dropped=drop, tops={c: list(v) for c, v in tops.items()},
                          base_macro_auc=round(m_base["macro_auc"], 4),
                          loo_macro_auc=round(m_lo["macro_auc"], 4),
                          delta=round(m_base["macro_auc"] - m_lo["macro_auc"], 4),
                          per_class=[ [round(m_base["per_class_auc"][c], 4), round(m_lo["per_class_auc"][c], 4)] for c in CLASS_ORDER ])
    print("SECTION 3 DONE")
    _done("3")


## 4) Figure 11 (Sokolow-Lyon by class + MI co-occurrence in HYP)


In [ ]:
_SEC = "4"
if _SEC in DONE and _SEC not in RERUN:
    print(f"=== 4) already completed (checkpoint) — skipping")
else:
    print("=== 4) FIGURE-11 (SOKOLOW + MI CO-OCCURRENCE) ===")
    mi = mi_indicator(df)
    # Left panel: Sokolow-Lyon index by class (HYP top single feature, §3.4).
    # (I_Q_Amplitude is ~all zeros — no Q wave delineated in lead I for most
    # records — so its boxplot collapses to a flat line; kept as 2nd choice.)
    plot_col = None
    for _c in ("Sokolow_Lyon_Index", "I_Q_Amplitude"):
        if _c in X.columns:
            plot_col = _c
            break
    if plot_col is None:
        _cand = [c for c in X.columns
                 if "sokolow" in str(c).lower() or "i_q" in str(c).lower() or "iq" in str(c).lower()]
        plot_col = _cand[0] if _cand else None
    if plot_col is None:
        print("WARNING: no Sokolow/I-Q amplitude column found — left panel skipped")
    else:
        _v = X[plot_col].dropna().to_numpy()
        _med = [float(X.loc[y == i, plot_col].median()) for i in range(5)]
        print(f">>> left panel column: {plot_col}  | zeros: {(_v == 0).mean():.1%} | median by class: "
              + ", ".join(f"{c} {_m:.4f}" for c, _m in zip(CLASS_ORDER, _med)))
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    if plot_col is not None:
        _is_sok = "sokolow" in plot_col.lower()
        data = [X.loc[y == i, plot_col].dropna() for i in range(5)]
        bp = axes[0].boxplot(data, showfliers=False, patch_artist=True)
        axes[0].set_xticks(range(1, 6))
        axes[0].set_xticklabels(CLASS_ORDER)
        for patch, col in zip(bp["boxes"], ["#4c72b0", "#dd8452", "#55a868", "#c44e52", "#8172b3"]):
            patch.set_facecolor(col)
        axes[0].set_ylabel(plot_col + (" (mV)" if (_is_sok or plot_col == "I_Q_Amplitude") else ""))
        axes[0].set_title("Sokolow–Lyon index by class" if _is_sok else "Lead-I Q amplitude by class")
    else:
        axes[0].axis("off")
    if mi is not None:
        frac = [float(mi[y == i].mean()) for i in range(5)]
        axes[1].bar(CLASS_ORDER, frac, color="#4c72b0")
        for j, v in enumerate(frac):
            axes[1].text(j, v + 0.01, f"{v:.0%}", ha="center", fontsize=9)
        axes[1].set_ylabel("share of records with >=1 MI superclass statement")
        axes[1].set_title("MI code co-occurrence by label")
        hyp_mi = float(mi[y == 4].mean())
        # self-check: MI-labelled records carry the MI statement that labelled them,
        # so their share must be ~100% with a complete superclass map
        if frac[1] < 0.90:
            print(f"WARNING: only {frac[1]:.0%} of MI-labelled records carry an MI statement (expect ~100%) — "
                  "superclass map may be incomplete; treat MI co-occurrence numbers with caution")
    else:
        axes[1].axis("off")
        frac = None
        hyp_mi = None
    fig.tight_layout()
    fig.savefig("figure11_cooccurrence.png", dpi=200)
    if hyp_mi is not None:
        print(f"HYP-labelled records with >=1 MI statement: {hyp_mi:.1%}")
    R["iq"] = dict(hyp_with_mi_pct=(round(hyp_mi * 100, 1) if hyp_mi is not None else None),
                   mi_share_by_class_pct=({c: round(float(f * 100), 1) for c, f in zip(CLASS_ORDER, frac)} if frac else None),
                   plot_column=plot_col)
    print("SECTION 4 DONE  (saved figure11_cooccurrence.png)")
    _done("4")


## 5) Unweighted LightGBM baseline (only difference vs main pipeline: no class weights)

In [ ]:
_SEC = "5"
if _SEC in DONE and _SEC not in RERUN:
    print(f"=== 5) already completed (checkpoint) — skipping")
else:
    if "baseline" not in R:
        raise SystemExit("Section 5 needs Section 1's result (R['baseline']) — Section 1 আগে complete করুন")
    m_base = R["baseline"]
    print("=== 5) UNWEIGHTED BASELINE ===")
    feats = feature_cols
    folds = df[FOLD_COL].to_numpy()
    n = len(df)
    probs_uw = np.zeros((n, 5))
    t0 = time.time()
    for k in range(1, 11):
        test = np.where(folds == k)[0]
        val = np.where(folds == (8 if k == 9 else 9))[0]
        train = np.setdiff1d(np.arange(n), np.union1d(test, val))
        m = lgb.LGBMClassifier(**PARAMS)
        m.fit(X.loc[train, feats].to_numpy(), y[train],   # NOTE: no sample_weight
              eval_set=[(X.loc[val, feats].to_numpy(), y[val])],
              eval_metric="multi_logloss",
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        probs_uw[test] = m.predict_proba(X.loc[test, feats].to_numpy())
        try:
            fa = roc_auc_score(y[test], probs_uw[test], multi_class='ovr', average='macro')
            fa_s = f"{fa:.4f}"
        except ValueError:
            fa_s = "n/a (class missing in fold)"
        print(f"  fold {k}: test AUC {fa_s}")
    print(f"  done in {(time.time()-t0)/60:.1f} min")
    m_uw = metrics(y, probs_uw)
    W_F1 = {c: m_base["per_class_f1"][c] for c in CLASS_ORDER}
    print(f"{'class':6s} {'F1 unw':>8s} {'F1 wtd':>8s} {'delta':>8s}")
    for c in CLASS_ORDER:
        d = m_uw["per_class_f1"][c] - W_F1[c]
        print(f"{c:6s} {m_uw['per_class_f1'][c]:8.4f} {W_F1[c]:8.3f} {d:+8.4f}")
    max_d = max(abs(m_uw["per_class_f1"][c] - W_F1[c]) for c in CLASS_ORDER if c != "HYP")
    print(f"max |delta F1| among non-HYP classes: {max_d:.4f}  (< 0.02 => use Template A in fix_05)")
    R["unweighted"] = dict(accuracy=round(m_uw["accuracy"], 4), macro_f1=round(m_uw["macro_f1"], 4),
                           macro_auc=round(m_uw["macro_auc"], 4),
                           per_class_f1={c: round(m_uw["per_class_f1"][c], 4) for c in CLASS_ORDER},
                           per_class_auc={c: round(m_uw["per_class_auc"][c], 4) for c in CLASS_ORDER},
                           max_nonhyp_delta_f1=round(max_d, 4))
    print("SECTION 5 DONE")
    _done("5")


## 6) ECE — top-label, raw vs inverse-reweighted (raw should reproduce 0.024; ⚠ stop if very different)

In [ ]:
_SEC = "6"
if _SEC in DONE and _SEC not in RERUN:
    print(f"=== 6) already completed (checkpoint) — skipping")
else:
    if not isinstance(globals().get("probs_oof"), np.ndarray):
        _p = os.path.join(CKPT_DIR, "probs_oof_weighted.csv")
        if not os.path.exists(_p):
            _p = "probs_oof_weighted.csv"
        if not os.path.exists(_p):
            raise SystemExit("Section 6 needs Section 1's OOF predictions (probs_oof_weighted.csv) — Section 1 আগে complete করুন")
        probs_oof = pd.read_csv(_p, index_col=0)[CLASS_ORDER].to_numpy()
        assert len(probs_oof) == len(df), "probs_oof rows mismatch vs loaded data — Section 1 আবার run করুন"
        print("reloaded probs_oof from checkpoint (Section 1 skipped in this session)")
    def top_label_ece(correct, conf, n_bins=10):
        edges = np.linspace(0, 1, n_bins + 1)
        ece = 0.0
        for m_ in range(n_bins):
            lo, hi = edges[m_], edges[m_ + 1]
            mask = (conf >= lo) & (conf <= hi) if m_ == 0 else (conf > lo) & (conf <= hi)
            if mask.sum() == 0:
                continue
            ece += mask.sum() / len(correct) * abs(correct[mask].mean() - conf[mask].mean())
        return float(ece)

    def classwise_ece(y, probs, n_bins=10):
        """Per-class ECE (bin by P(class), compare to 1{true==class}), averaged over classes."""
        edges = np.linspace(0, 1, n_bins + 1)
        tot = 0.0
        for c in range(probs.shape[1]):
            conf = probs[:, c]
            correct = (y == c).astype(float)
            ece_c = 0.0
            for m_ in range(n_bins):
                lo, hi = edges[m_], edges[m_ + 1]
                mask = (conf >= lo) & (conf <= hi) if m_ == 0 else (conf > lo) & (conf <= hi)
                if mask.sum() == 0:
                    continue
                ece_c += mask.sum() / len(conf) * abs(correct[mask].mean() - conf[mask].mean())
            tot += ece_c
        return float(tot / probs.shape[1])

    print("=== 6) ECE ===")
    pred = probs_oof.argmax(axis=1)
    ece_raw = top_label_ece((pred == y).astype(float), probs_oof.max(axis=1))
    ece_cls = classwise_ece(y, probs_oof)
    folds = df[FOLD_COL].to_numpy()
    p_corr = probs_oof.copy()
    for k in range(1, 11):
        in_test = folds == k
        val_fold = 8 if k == 9 else 9
        tr = np.isin(folds, [f for f in range(1, 11) if f not in (k, val_fold)])
        N = int(tr.sum())
        counts = np.bincount(y[tr], minlength=5)
        w = N / (5 * np.maximum(counts, 1))
        p_corr[in_test] = probs_oof[in_test] / w
        p_corr[in_test] /= p_corr[in_test].sum(axis=1, keepdims=True)
    pred_c = p_corr.argmax(axis=1)
    ece_rew = top_label_ece((pred_c == y).astype(float), p_corr.max(axis=1))
    print(f"Top-label ECE raw (balanced-weight probabilities): {ece_raw:.4f}")
    print(f"Top-label ECE after inverse reweighting:           {ece_rew:.4f}")
    print(f"Class-wise ECE (average over 5 classes):            {ece_cls:.4f}   <- the paper's 0.024 is likely THIS definition")
    R["ece"] = dict(top_label_raw=round(ece_raw, 4), top_label_reweighted=round(ece_rew, 4),
                    classwise=round(ece_cls, 4))
    print("SECTION 6 DONE")
    _done("6")


## 7) Multi-label (CinC-style) evaluation — micro-F1@0.47, sample F1, macro AUC (paper: 0.792 / 0.932)

In [ ]:
_SEC = "7"
if _SEC in DONE and _SEC not in RERUN:
    print(f"=== 7) already completed (checkpoint) — skipping")
else:
    print("=== 7) MULTI-LABEL (CinC-STYLE) ===")
    try:
        if ML_FILE is not None:
            ml = pd.read_csv(ML_FILE)
            idc = ml.columns[0]
            ml = ml.set_index(idc)
            common = ml.index.intersection(df[ID_COL])
            idx = df.index[df[ID_COL].isin(common)].to_numpy()
            Y = ml.loc[common, CLASS_ORDER].to_numpy(dtype=int)
            src = f"ML_FILE: {os.path.basename(ML_FILE)}"
        else:
            _db = db_indexed()
            if _db is None:
                raise SystemExit("SECTION 7 SKIPPED: no ML_FILE and no usable PTBXL_DB to derive the multi-label ground truth")
            dbx, scpc = _db
            scmap = load_superclass_map()
            Y = np.array([_codes_to_flags(s, scmap) for s in dbx[scpc].reindex(df[ID_COL])], dtype=int)
            idx = np.arange(len(df))
            src = "derived from ptbxl_database.csv scp_codes (CinC-2020 superclass rule)"
            matched = int(df[ID_COL].isin(dbx.index).sum())
            if matched < 0.95 * len(df):
                print("WARNING: id join failed — feature ids sample:", df[ID_COL].head(3).tolist(),
                      "| database ids sample:", list(dbx.index[:3]))
        print(f"records: {len(idx)}  | ground-truth source: {src}")
        ysub = y[idx]
        single = np.array([(ysub == i).mean() for i in range(5)])
        multi = Y.mean(0)
        print("single-label prevalence (cohort):", {c: round(float(v), 3) for c, v in zip(CLASS_ORDER, single)})
        print("multi-label prevalence (cohort):", {c: round(float(v), 3) for c, v in zip(CLASS_ORDER, multi)})
        bad = [CLASS_ORDER[i] for i in range(1, 5) if multi[i] + 1e-9 < single[i]]
        if bad:
            print("WARNING: multi-label prevalence of", bad, "is LOWER than single-label — id join may be misaligned; treat these numbers with caution")
        folds2 = df[FOLD_COL].to_numpy()[idx]
        train = np.arange(len(idx))[np.isin(folds2, range(1, 9))]
        val = np.where(folds2 == 9)[0]
        test = np.where(folds2 == 10)[0]
        p_test, p_val, thresholds = np.zeros((len(test), 5)), np.zeros((len(val), 5)), []
        BIN = dict(PARAMS)
        BIN["objective"] = "binary"
        BIN.pop("num_class", None)   # binary objective is incompatible with num_class
        for ci, c in enumerate(CLASS_ORDER):
            yb = Y[train, ci]
            sw = np.ones(len(yb))
            sw[yb == 1] = len(yb) / (2 * max(yb.sum(), 1))
            sw[yb == 0] = len(yb) / (2 * max(len(yb) - yb.sum(), 1))
            m = lgb.LGBMClassifier(**BIN)
            m.fit(X.loc[train, feature_cols].to_numpy(), yb, sample_weight=sw,
                  eval_set=[(X.loc[val, feature_cols].to_numpy(), Y[val, ci])],
                  eval_metric="binary_logloss",
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
            p_val[:, ci] = m.predict_proba(X.loc[val, feature_cols].to_numpy())[:, 1]
            p_test[:, ci] = m.predict_proba(X.loc[test, feature_cols].to_numpy())[:, 1]
            best_t, best_f = 0.5, -1.0
            for t in np.arange(0.05, 0.96, 0.01):
                f = f1_score(Y[val, ci], (p_val[:, ci] >= t).astype(int), zero_division=1)
                if f > best_f:
                    best_t, best_f = float(t), f
            thresholds.append(best_t)
            print(f"  {c:5s}: best val threshold t={best_t:.2f} (val F1 {best_f:.3f})")
        pred47 = (p_test >= 0.47).astype(int)
        tp = ((pred47 == 1) & (Y[test] == 1)).sum(); fp = ((pred47 == 1) & (Y[test] == 0)).sum(); fn = ((pred47 == 0) & (Y[test] == 1)).sum()
        micro = 2 * tp / (2 * tp + fp + fn)
        thr = np.array(thresholds)
        predp = (p_test >= thr).astype(int)
        tp_r = ((predp == 1) & (Y[test] == 1)).sum(1); fp_r = ((predp == 1) & (Y[test] == 0)).sum(1); fn_r = ((predp == 0) & (Y[test] == 1)).sum(1)
        sample = float(np.where((tp_r + fp_r + fn_r) == 0, 1.0, 2 * tp_r / (2 * tp_r + fp_r + fn_r)).mean())
        aucs_ml = {}
        for ci, c in enumerate(CLASS_ORDER):
            try:
                aucs_ml[c] = float(roc_auc_score(Y[test, ci], p_test[:, ci]))
            except ValueError:
                aucs_ml[c] = float("nan")
        macro_auc_ml = float(np.nanmean(list(aucs_ml.values())))
        print(f"micro-F1 at uniform t=0.47:            {micro:.4f}   (paper F-max: 0.792)")
        print(f"sample-level F1 (per-label thresholds): {sample:.4f}   (best published [3]: 0.823)")
        print("multi-label per-class AUC (fold-10 test):", {c: round(v, 4) for c, v in aucs_ml.items()})
        print(f"multi-label macro AUC:                  {macro_auc_ml:.4f}   (paper: 0.932)")
        R["cinc"] = dict(source=src, micro_f1_t047=round(float(micro), 4), sample_f1=round(sample, 4),
                         macro_auc=round(macro_auc_ml, 4),
                         per_class_auc={c: (round(v, 4) if v == v else None) for c, v in aucs_ml.items()},
                         multi_label_prevalence={c: round(float(v), 3) for c, v in zip(CLASS_ORDER, multi)},
                         thresholds={c: t for c, t in zip(CLASS_ORDER, thresholds)})
        print("SECTION 7 DONE")
    except SystemExit:
        raise
    except Exception as e:
        import traceback
        traceback.print_exc()
        R["cinc"] = f"FAILED: {type(e).__name__}: {str(e)[:200]}"
    _c7 = R.get("cinc")
    if isinstance(_c7, dict):
        _done("7")
    else:
        print("Section 7 not checkpointed (", _c7, ") — ঠিক করে এই cell আবার run করুন")


## 8) SHAP on all 10 folds + rank stability

In [ ]:
_SEC = "8"
if _SEC in DONE and _SEC not in RERUN:
    print(f"=== 8) already completed (checkpoint) — skipping")
else:
    print("=== 8) SHAP 10-FOLD STABILITY ===")
    try:
        import shap
    except ImportError:
        raise SystemExit("pip install shap   (then re-run this cell)")
    folds = df[FOLD_COL].to_numpy()
    n = len(df)
    imps = []
    t0 = time.time()
    for k in range(1, 11):
        test = np.where(folds == k)[0]
        val = np.where(folds == (8 if k == 9 else 9))[0]
        train = np.setdiff1d(np.arange(n), np.union1d(test, val))
        m = lgb.LGBMClassifier(**PARAMS)
        m.fit(X.loc[train, feature_cols].to_numpy(), y[train], sample_weight=class_sample_weights(y[train]),
              eval_set=[(X.loc[val, feature_cols].to_numpy(), y[val])],
              eval_metric="multi_logloss",
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        try:
            sv = shap.TreeExplainer(m).shap_values(X.loc[test, feature_cols].to_numpy())
        except Exception:
            sv = shap.TreeExplainer(m.booster_).shap_values(X.loc[test, feature_cols].to_numpy())
        imps.append(sum(np.abs(s) for s in sv).mean(axis=0))
        print(f"  fold {k} done ({(time.time()-t0)/60:.1f} min)")
    imps = np.array(imps)
    sp = np.zeros((10, 10))
    for a in range(10):
        for b in range(10):
            sp[a, b] = spearmanr(imps[a], imps[b])[0]
    mean_rho = float(sp[~np.eye(10, dtype=bool)].mean() / 2)
    ranks = np.argsort(np.argsort(-imps, axis=1), axis=1) + 1   # 1 = most important fold-wise
    top10_idx = np.argsort(-imps.mean(axis=0))[:10]
    mean_rank_top10 = float(ranks[:, top10_idx].mean())
    print(f"mean pairwise Spearman rho (10 folds): {mean_rho:.4f}")
    print(f"mean rank of overall top-10 features:  {mean_rank_top10:.2f}  (1 = always top)")
    print("fold-10 top-5 (paper Table 8 basis): " + ", ".join(np.array(feature_cols)[np.argsort(-imps[9])][:5]))
    R["shap"] = dict(mean_pairwise_spearman=round(mean_rho, 4), mean_rank_top10=round(mean_rank_top10, 2),
                     fold10_top5=list(np.array(feature_cols)[np.argsort(-imps[9])][:5]))
    print("SECTION 8 DONE")
    _done("8")


## 8b) Unambiguous single-label subset — paper Section 4.4 (needs database)

In [ ]:
_SEC = "8b"
if _SEC in DONE and _SEC not in RERUN:
    print(f"=== 8b) already completed (checkpoint) — skipping")
else:
    print("=== 8b) UNAMBIGUOUS SINGLE-LABEL SUBSET ===")
    try:
        _db = db_indexed()
        if _db is None:
            raise SystemExit("SECTION 8b SKIPPED: no usable PTBXL_DB")
        dbx, scpc = _db
        scmap = load_superclass_map()
        sets = {}
        for _idv, _s in zip(df[ID_COL], dbx[scpc].reindex(df[ID_COL])):
            cs = set()
            try:
                for _c in ast.literal_eval(_s):
                    _sc = scmap.get(str(_c).strip())
                    if _sc:
                        cs.add(_sc)
            except Exception:
                pass
            sets[_idv] = cs
        n_multi = n_nondiag = n_mismatch = 0
        keep = np.ones(len(df), bool)
        for i, _idv in enumerate(df[ID_COL]):
            cs = sets[_idv]
            lab = CLASS_ORDER[y[i]]
            if len(cs) == 0:
                n_nondiag += 1; keep[i] = False
            elif len(cs) > 1:
                n_multi += 1; keep[i] = False
            elif lab not in cs:
                n_mismatch += 1; keep[i] = False
        sub_df = df.loc[keep].reset_index(drop=True)  # v2.5 fix: run_cv uses positional .loc
        ysub = y[keep]
        n_sub = int(keep.sum())
        sub_counts = pd.Series(ysub).value_counts().reindex(list(range(5)))
        print(f"excluded: multi-superclass {n_multi} | no-diagnostic {n_nondiag} | single-superclass-mismatch {n_mismatch}")
        print(f"unambiguous subset: {n_sub} records, patients {sub_df[PATIENT_COL].nunique()}")
        print("subset class composition:", {c: f"{int(n)} ({n / n_sub * 100:.1f}%)"
              for c, n in zip(CLASS_ORDER, sub_counts.tolist())})
        if n_sub < 10000 or not set(range(5)).issubset(set(np.unique(ysub))):
            print("WARNING: subset unexpectedly small or a class is missing — check the database before using these numbers")
        Xsub = X.loc[keep].reset_index(drop=True)
        probs_sub = run_cv(sub_df, Xsub, ysub, feature_cols, tag="unamb")
        m_sub = metrics(ysub, probs_sub)
        full = R.get("baseline", {})
        d = {k: m_sub[k] - full[k] for k in ("accuracy", "macro_f1", "macro_auc")}
        print(f"subset pooled OOF: acc {m_sub['accuracy']:.4f} (delta {d['accuracy']:+.3f}) | "
              f"macroF1 {m_sub['macro_f1']:.4f} (delta {d['macro_f1']:+.3f}) | "
              f"macroAUC {m_sub['macro_auc']:.4f} (delta {d['macro_auc']:+.3f})")
        print("subset per-class F1:", {c: round(v, 4) for c, v in m_sub["per_class_f1"].items()})
        pd.DataFrame(probs_sub, columns=CLASS_ORDER, index=sub_df[ID_COL]).to_csv("probs_oof_unambiguous.csv", index_label=ID_COL)
        R["unambiguous_subset"] = dict(
            n_multi=int(n_multi), n_nondiag=int(n_nondiag), n_mismatch=int(n_mismatch),
            subset_n=n_sub, subset_patients=int(sub_df[PATIENT_COL].nunique()),
            subset_composition={c: {"n": int(n), "pct": round(float(n) / n_sub * 100, 1)}
                                for c, n in zip(CLASS_ORDER, sub_counts.tolist())},
            accuracy=round(m_sub["accuracy"], 4), macro_f1=round(m_sub["macro_f1"], 4),
            macro_auc=round(m_sub["macro_auc"], 4),
            delta_vs_full={k: round(v, 4) for k, v in d.items()},
            per_class_f1={c: round(v, 4) for c, v in m_sub["per_class_f1"].items()},
            per_class_auc={c: round(v, 4) for c, v in m_sub["per_class_auc"].items()})
        print("SECTION 8b DONE")
    except SystemExit:
        raise
    except Exception as e:
        import traceback
        traceback.print_exc()
        R["unambiguous_subset"] = f"FAILED: {type(e).__name__}: {str(e)[:200]}"
    _c8b = R.get("unambiguous_subset")
    if isinstance(_c8b, dict):
        _done("8b")
    else:
        print("Section 8b not checkpointed (", _c8b, ") — ঠিক করে এই cell আবার run করুন")


## 8c) Figure regeneration — Fig 3 / Fig 5 / subset-CM from checkpoints (no retraining)

In [ ]:
_SEC = "8c"
if _SEC in DONE and _SEC not in RERUN:
    print(f"=== 8c) already completed (checkpoint) — skipping")
else:
    print("=== 8c) FIGURE REGENERATION ===")
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
    _made = {}
    try:
        _pw = os.path.join(CKPT_DIR, "probs_oof_weighted.csv")
        if not os.path.exists(_pw):
            _pw = "probs_oof_weighted.csv"
        if not os.path.exists(_pw):
            raise SystemExit("SECTION 8c SKIPPED: probs_oof_weighted.csv not found — run section 1 first (or keep its checkpoint)")
        _P = pd.read_csv(_pw).set_index(pd.read_csv(_pw, nrows=0).columns[0])
        _P = _P.reindex(df[ID_COL])
        if _P[CLASS_ORDER].isna().any().any():
            raise SystemExit("SECTION 8c SKIPPED: probability file ids do not cover the cohort — re-run section 1")
        _probs = _P[CLASS_ORDER].to_numpy()
        _pred = _probs.argmax(axis=1)
        _acc = float(accuracy_score(y, _pred)); _mf1 = float(f1_score(y, _pred, average="macro"))
        _mauc = float(roc_auc_score(y, _probs, multi_class="ovr", average="macro"))
        print(f">>> OOF check: acc {_acc:.4f} | macroF1 {_mf1:.4f} | macroAUC {_mauc:.4f} (expect 0.7613 / 0.7010 / 0.9348)")

        def _plot_cm(_yy, _pp, _title, _fname, _cmap):
            _cc = confusion_matrix(_yy, _pp, labels=list(range(5)))
            _pct = _cc / _cc.sum(axis=1, keepdims=True)
            _fig, _ax = plt.subplots(figsize=(9.7, 7.5), dpi=100)
            _im = _ax.imshow(_pct, cmap=_cmap, vmin=0, vmax=1)
            for _i in range(5):
                for _j in range(5):
                    _col = "white" if _pct[_i, _j] > 0.5 else "black"
                    _ax.text(_j, _i, f"{_pct[_i, _j]:.1%}\n({_cc[_i, _j]})", ha="center", va="center", fontsize=11, color=_col)
            _ax.set_xticks(range(5)); _ax.set_xticklabels(CLASS_ORDER, fontsize=12)
            _ax.set_yticks(range(5)); _ax.set_yticklabels(CLASS_ORDER, fontsize=12)
            _ax.set_xlabel("Predicted", fontsize=13); _ax.set_ylabel("True", fontsize=13)
            _ax.set_title(_title, fontsize=13)
            _fig.colorbar(_im, ax=_ax, fraction=0.046, pad=0.04)
            _fig.tight_layout(); _fig.savefig(_fname, dpi=150); plt.close(_fig)

        _plot_cm(y, _pred,
                 f"Pooled out-of-fold confusion matrix (n = {len(y):,})\nbalanced · acc = {_acc:.3f} · macro-F1 = {_mf1:.3f} · macro-AUC = {_mauc:.3f}",
                 "figure3_confusion_matrix.png", "Blues")
        _made["figure3"] = "figure3_confusion_matrix.png"
        print(">>> saved figure3_confusion_matrix.png")

        _conf = _probs.max(axis=1); _corr = (_pred == y).astype(float)
        _edges = np.linspace(0, 1, 11)
        _acc_b, _conf_b, _cnt_b = [], [], []
        for _b in range(10):
            _m = (_conf > _edges[_b]) & (_conf <= _edges[_b + 1]) if _b else (_conf >= 0.0) & (_conf <= _edges[1])
            _cnt_b.append(int(_m.sum()))
            _acc_b.append(float(_corr[_m].mean()) if _m.sum() else 0.0)
            _conf_b.append(float(_conf[_m].mean()) if _m.sum() else float((_edges[_b] + _edges[_b + 1]) / 2))
        _cnt_b = np.array(_cnt_b, dtype=float)
        _ece = float((np.abs(np.array(_acc_b) - np.array(_conf_b)) * _cnt_b / _cnt_b.sum()).sum())
        _fig, _ax = plt.subplots(figsize=(8.5, 7.5), dpi=100)
        _x = (_edges[:-1] + _edges[1:]) / 2
        _ax.plot([0, 1], [0, 1], "k--", lw=1.5, label="perfect calibration")
        _ax.bar(_x, _acc_b, width=0.08, color="#6baed6", label="model")
        _ax.scatter(_conf_b, _acc_b, s=30 + 400 * _cnt_b / max(_cnt_b.max(), 1), color="#084594", zorder=3)
        _ax.set_xlim(0, 1.0); _ax.set_ylim(0, 1.0)
        _ax.set_xlabel("Mean predicted confidence", fontsize=12)
        _ax.set_ylabel("Observed accuracy", fontsize=12)
        _ax.set_title(f"Reliability diagram — ECE = {_ece:.4f}", fontsize=13)
        _ax.legend(fontsize=11)
        _fig.tight_layout(); _fig.savefig("figure5_reliability.png", dpi=150); plt.close(_fig)
        _made["figure5"] = "figure5_reliability.png"
        print(f">>> saved figure5_reliability.png (ECE {_ece:.4f}; §6 said {R.get('ece', {}).get('top_label_raw', 'n/a') if isinstance(R.get('ece'), dict) else 'n/a'})")

        _pu = os.path.join(CKPT_DIR, "probs_oof_unambiguous.csv")
        if not os.path.exists(_pu):
            _pu = "probs_oof_unambiguous.csv"
        if os.path.exists(_pu):
            _U = pd.read_csv(_pu); _uid = _U.columns[0]; _U = _U.set_index(_uid)
            _id2y = dict(zip(df[ID_COL].tolist(), y.tolist()))
            _yu = np.array([_id2y[i] for i in _U.index])
            _p8 = _U[CLASS_ORDER].to_numpy(); _pr = _p8.argmax(axis=1)
            _a = float(accuracy_score(_yu, _pr)); _f = float(f1_score(_yu, _pr, average="macro"))
            try:
                _au = float(roc_auc_score(_yu, _p8, multi_class="ovr", average="macro"))
            except ValueError:
                _au = float("nan")
            _plot_cm(_yu, _pr,
                     f"Unambiguous single-label subset (n = {len(_yu):,})\nacc = {_a:.3f} · macro-F1 = {_f:.3f} · macro-AUC = {_au:.3f}",
                     "figure4_subset_cm.png", "Greens")
            _made["figure4_subset"] = "figure4_subset_cm.png"
            print(f">>> saved figure4_subset_cm.png (n={len(_yu)}, acc {_a:.4f}, macroAUC {_au:.4f})")
        else:
            print(">>> subset probs not found (§8b) — Fig 4 regeneration skipped")
        R["figures"] = dict(_made, ece_fig5=round(_ece, 4),
                            oof_check=dict(acc=round(_acc, 4), macro_f1=round(_mf1, 4), macro_auc=round(_mauc, 4)))
        print("SECTION 8c DONE")
    except SystemExit:
        raise
    except Exception as e:
        import traceback
        traceback.print_exc()
        R["figures"] = f"FAILED: {type(e).__name__}: {str(e)[:200]}"
    _c8c = R.get("figures")
    if isinstance(_c8c, dict):
        _done("8c")
    else:
        print("Section 8c not checkpointed (", _c8c, ") — ঠিক করে এই cell আবার run করুন")


## 8d) No-HRV ablation — full CV without the 6 HRV features (~20 min)

In [ ]:
_SEC = "8d"
if _SEC in DONE and _SEC not in RERUN:
    print(f"=== 8d) already completed (checkpoint) — skipping")
else:
    print("=== 8d) NO-HRV ABLATION ===")
    try:
        _hrv = [c for c in feature_cols if c.startswith("HRV_")]
        if "N_RPeaks" in feature_cols and "N_RPeaks" not in _hrv:
            _hrv.append("N_RPeaks")
        print("HRV-family columns detected:", sorted(_hrv))
        if len(_hrv) != 6:
            _cands = sorted(set([c for c in feature_cols
                                 if "hrv" in c.lower() or "rpeak" in c.lower() or c.startswith("HRV_")]))
            raise SystemExit(f"SECTION 8d SKIPPED: expected 6 HRV columns, found {len(_hrv)} — candidates: {_cands}")
        _p8d = run_cv(df, X, y, feature_cols, exclude=_hrv, tag="noHRV")
        _m8d = metrics(y, _p8d)
        _base = R.get("baseline", {})
        _dA = float(_m8d["macro_auc"]) - float(_base.get("macro_auc", float("nan")))
        _dF = float(_m8d["macro_f1"]) - float(_base.get("macro_f1", float("nan")))
        print(f"no-HRV pooled OOF: acc {_m8d['accuracy']:.4f} | macroF1 {_m8d['macro_f1']:.4f} | macroAUC {_m8d['macro_auc']:.4f} "
              f"(delta AUC {_dA:+.4f}, delta F1 {_dF:+.4f} vs full 543-feature model)")
        R["ablation_nohrv"] = dict(excluded=sorted(_hrv), n_features=len(feature_cols) - 6,
                                   accuracy=round(float(_m8d["accuracy"]), 4),
                                   macro_f1=round(float(_m8d["macro_f1"]), 4),
                                   macro_auc=round(float(_m8d["macro_auc"]), 4),
                                   delta_auc_vs_full=round(_dA, 4),
                                   delta_f1_vs_full=round(_dF, 4))
        print("SECTION 8d DONE")
    except SystemExit:
        raise
    except Exception as e:
        import traceback
        traceback.print_exc()
        R["ablation_nohrv"] = f"FAILED: {type(e).__name__}: {str(e)[:200]}"
    _c8d = R.get("ablation_nohrv")
    if isinstance(_c8d, dict):
        _done("8d")
    else:
        print("Section 8d not checkpointed (", _c8d, ") — ঠিক করে এই cell আবার run করুন")


## 8e) Per-fold mean±SD from checkpoint probs (~1 min, no retraining)

In [ ]:
_SEC = "8e"
if _SEC in DONE and _SEC not in RERUN:
    print(f"=== 8e) already completed (checkpoint) — skipping")
else:
    print("=== 8e) PER-FOLD MEAN±SD (checkpoint probs, no retraining) ===")
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
    try:
        _pw = os.path.join(CKPT_DIR, "probs_oof_weighted.csv")
        if not os.path.exists(_pw):
            _pw = "probs_oof_weighted.csv"
        if not os.path.exists(_pw):
            raise SystemExit("SECTION 8e SKIPPED: probs_oof_weighted.csv not found")
        _P = pd.read_csv(_pw).set_index(pd.read_csv(_pw, nrows=0).columns[0])
        _P = _P.reindex(df[ID_COL])
        if _P[CLASS_ORDER].isna().any().any():
            raise SystemExit("SECTION 8e SKIPPED: probability file ids do not cover the cohort")
        _probs = _P[CLASS_ORDER].to_numpy()
        _folds = df[FOLD_COL].to_numpy()
        _acc, _mf1, _wf1, _mauc, _pcf1 = [], [], [], [], {c: [] for c in CLASS_ORDER}
        for _k in range(1, 11):
            _m = _folds == _k
            _yt, _pt = y[_m], _probs[_m]
            _pr = _pt.argmax(axis=1)
            _acc.append(float(accuracy_score(_yt, _pr)))
            _mf1.append(float(f1_score(_yt, _pr, average="macro")))
            _wf1.append(float(f1_score(_yt, _pr, average="weighted")))
            try:
                _mauc.append(float(roc_auc_score(_yt, _pt, multi_class="ovr", average="macro")))
            except ValueError:
                _mauc.append(float("nan"))
            for _ii, _c in enumerate(CLASS_ORDER):
                _pcf1[_c].append(float(f1_score((_yt == _ii).astype(int), (_pr == _ii).astype(int))))
        def _ms(_v):
            _a = np.array(_v, dtype=float)
            return dict(mean=round(float(np.nanmean(_a)), 4), sd=round(float(np.nanstd(_a, ddof=1)), 4))
        R["perfold"] = dict(n_folds=10,
                            accuracy=_ms(_acc), macro_f1=_ms(_mf1), weighted_f1=_ms(_wf1), macro_auc=_ms(_mauc),
                            per_class_f1={_c: _ms(_v) for _c, _v in _pcf1.items()},
                            per_fold=dict(accuracy=[round(v, 4) for v in _acc],
                                          macro_f1=[round(v, 4) for v in _mf1],
                                          weighted_f1=[round(v, 4) for v in _wf1],
                                          macro_auc=[round(v, 4) for v in _mauc]))
        _pf = R["perfold"]
        print(f"per-fold (n=10): acc {_pf['accuracy']['mean']:.4f}±{_pf['accuracy']['sd']:.4f} | "
              f"macroF1 {_pf['macro_f1']['mean']:.4f}±{_pf['macro_f1']['sd']:.4f} | "
              f"wF1 {_pf['weighted_f1']['mean']:.4f}±{_pf['weighted_f1']['sd']:.4f} | "
              f"macroAUC {_pf['macro_auc']['mean']:.4f}±{_pf['macro_auc']['sd']:.4f}")
        print("per-class F1:", {c: f"{v['mean']:.4f}±{v['sd']:.4f}" for c, v in _pf["per_class_f1"].items()})
        print("SECTION 8e DONE")
    except SystemExit:
        raise
    except Exception as e:
        import traceback
        traceback.print_exc()
        R["perfold"] = f"FAILED: {type(e).__name__}: {str(e)[:200]}"
    _c8e = R.get("perfold")
    if isinstance(_c8e, dict):
        _done("8e")
    else:
        print("Section 8e not checkpointed (", _c8e, ") — ঠিক করে এই cell আবার run করুন")


## 8g) Reviewer re-runs — multi-label Fmax + CI (8g1), SHAP stability bug-fixed (8g2), six-family ablation clean run (8g3)

In [ ]:
# =============================================================================
# 8g) Reviewer re-runs — paste as ONE new code cell after Section 8e in the v2.5 notebook
#     (needs the notebook objects: df, X, y, feature_cols, FOLD_COL, PATIENT_COL, ID_COL,
#      PARAMS, class_sample_weights, run_cv, metrics, CKPT_DIR, R, DONE, RERUN, _done,
#      CLASS_ORDER, ML_FILE / db_indexed / load_superclass_map / _codes_to_flags from Section 7)
#
#   8g-0  dump feature list                                   (~1 s)
#   8g-1  multi-label: true Fmax (threshold chosen on fold 9), fold-10 n, 2,000 patient bootstrap CI   (~5 min)
#   8g-2  SHAP stability, bug-fixed (10 folds)                (~30 min)
#   8g-3  family ablation, 6 families alone, clean run        (~1.5-2 h)
# Each sub-section checkpoints separately (keys 8g1, 8g2, 8g3) so an interrupted run resumes.
# =============================================================================
import numpy as np, pandas as pd, lightgbm as lgb, os, json, time
from sklearn.metrics import roc_auc_score, f1_score
from scipy.stats import spearmanr

# ---------- 8g-0 feature list ----------
pd.Series(feature_cols).to_csv(os.path.join(CKPT_DIR, "feature_cols.csv"), index=False, header=["feature"])
print(f"8g-0: {len(feature_cols)} feature names saved to feature_cols.csv")

# ---------- 8g-1 multi-label Fmax + CI ----------
if "8g1" in DONE and "8g1" not in RERUN:
    print("=== 8g-1 already completed — skipping")
else:
    print("=== 8g-1) MULTI-LABEL: Fmax on fold-9 threshold, fold-10 n, bootstrap CI ===")
    if ML_FILE is not None:
        ml = pd.read_csv(ML_FILE); ml = ml.set_index(ml.columns[0])
        common = ml.index.intersection(df[ID_COL]); idx = df.index[df[ID_COL].isin(common)].to_numpy()
        Y = ml.loc[common, CLASS_ORDER].to_numpy(dtype=int)
    else:
        dbx, scpc = db_indexed(); scmap = load_superclass_map()
        Y = np.array([_codes_to_flags(s, scmap) for s in dbx[scpc].reindex(df[ID_COL])], dtype=int); idx = np.arange(len(df))
    folds2 = df[FOLD_COL].to_numpy()[idx]; pats = df[PATIENT_COL].to_numpy()[idx]
    train = np.arange(len(idx))[np.isin(folds2, range(1, 9))]; val = np.where(folds2 == 9)[0]; test = np.where(folds2 == 10)[0]
    BIN = dict(PARAMS); BIN["objective"] = "binary"; BIN.pop("num_class", None)
    p_test, p_val = np.zeros((len(test), 5)), np.zeros((len(val), 5))
    for ci, c in enumerate(CLASS_ORDER):
        yb = Y[train, ci]; sw = np.ones(len(yb))
        sw[yb == 1] = len(yb) / (2 * max(yb.sum(), 1)); sw[yb == 0] = len(yb) / (2 * max(len(yb) - yb.sum(), 1))
        m = lgb.LGBMClassifier(**BIN)
        m.fit(X.loc[train, feature_cols].to_numpy(), yb, sample_weight=sw,
              eval_set=[(X.loc[val, feature_cols].to_numpy(), Y[val, ci])], eval_metric="binary_logloss",
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        p_val[:, ci] = m.predict_proba(X.loc[val, feature_cols].to_numpy())[:, 1]
        p_test[:, ci] = m.predict_proba(X.loc[test, feature_cols].to_numpy())[:, 1]
    def micro_f1(P, Yt, t):
        pr = (P >= t).astype(int); tp = ((pr == 1) & (Yt == 1)).sum(); fp = ((pr == 1) & (Yt == 0)).sum(); fn = ((pr == 0) & (Yt == 1)).sum()
        return 2 * tp / max(2 * tp + fp + fn, 1)
    def sample_f1(P, Yt, t):
        pr = (P >= t).astype(int); tp = ((pr == 1) & (Yt == 1)).sum(1); fp = ((pr == 1) & (Yt == 0)).sum(1); fn = ((pr == 0) & (Yt == 1)).sum(1)
        return float(np.where((tp + fp + fn) == 0, 1.0, 2 * tp / np.maximum(2 * tp + fp + fn, 1)).mean())
    grid = np.arange(0.05, 0.96, 0.01)
    # thresholds chosen on fold 9 ONLY
    t_micro = float(grid[np.argmax([micro_f1(p_val, Y[val], t) for t in grid])])
    t_sample = float(grid[np.argmax([sample_f1(p_val, Y[val], t) for t in grid])])
    thr_lab = np.array([float(grid[np.argmax([f1_score(Y[val, ci], (p_val[:, ci] >= t).astype(int), zero_division=1) for t in grid])]) for ci in range(5)])
    res = dict(n_test_fold10=int(len(test)), n_patients_fold10=int(len(np.unique(pats[test]))),
               t_micro_fold9=t_micro, micro_f1_at_t_micro=round(micro_f1(p_test, Y[test], t_micro), 4),
               t_sample_fold9=t_sample, sample_f1_uniform_at_t_sample=round(sample_f1(p_test, Y[test], t_sample), 4),
               micro_f1_fixed_047=round(micro_f1(p_test, Y[test], 0.47), 4),
               sample_f1_perlabel_thr=round(sample_f1(p_test, Y[test], thr_lab), 4),
               macro_auc=round(float(np.mean([roc_auc_score(Y[test, ci], p_test[:, ci]) for ci in range(5)])), 4))
    # patient-level bootstrap on fold-10 test
    rng = np.random.default_rng(42); up = np.unique(pats[test]); pos = {p: np.where(pats[test] == p)[0] for p in up}
    b_auc, b_s, b_m = [], [], []
    for _ in range(2000):
        sel = np.concatenate([pos[p] for p in rng.choice(up, len(up), replace=True)])
        Yt, Pt = Y[test][sel], p_test[sel]
        try: b_auc.append(np.mean([roc_auc_score(Yt[:, ci], Pt[:, ci]) for ci in range(5)]))
        except ValueError: pass
        b_s.append(sample_f1(Pt, Yt, thr_lab)); b_m.append(micro_f1(Pt, Yt, t_micro))
    ci = lambda a: [round(float(np.percentile(a, 2.5)), 4), round(float(np.percentile(a, 97.5)), 4)]
    res.update(ci_macro_auc=ci(b_auc), ci_sample_f1=ci(b_s), ci_micro_f1=ci(b_m))
    print(json.dumps(res, indent=1)); R["cinc_fmax_ci"] = res; _done("8g1")

# ---------- 8g-2 SHAP stability (bug-fixed) ----------
if "8g2" in DONE and "8g2" not in RERUN:
    print("=== 8g-2 already completed — skipping")
else:
    print("=== 8g-2) SHAP STABILITY, 10 folds ===")
    import shap
    folds = df[FOLD_COL].to_numpy(); n = len(df); imps = []; t0 = time.time()
    for k in range(1, 11):
        test = np.where(folds == k)[0]; val = np.where(folds == (8 if k == 9 else 9))[0]
        train = np.setdiff1d(np.arange(n), np.union1d(test, val))
        m = lgb.LGBMClassifier(**PARAMS)
        m.fit(X.loc[train, feature_cols].to_numpy(), y[train], sample_weight=class_sample_weights(y[train]),
              eval_set=[(X.loc[val, feature_cols].to_numpy(), y[val])], eval_metric="multi_logloss",
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        sv = shap.TreeExplainer(m).shap_values(X.loc[test, feature_cols].to_numpy())
        sv = np.stack(sv, -1) if isinstance(sv, list) else np.asarray(sv)      # n x p x K  (FIX)
        assert sv.shape[1] == len(feature_cols), sv.shape
        imps.append(np.abs(sv).mean(axis=(0, 2)))                              # length p   (FIX)
        print(f"  fold {k} done ({(time.time()-t0)/60:.1f} min) top-3: {list(np.array(feature_cols)[np.argsort(-imps[-1])][:3])}")
    imps = np.array(imps)                                                      # 10 x p
    rhos = [spearmanr(imps[a], imps[b])[0] for a in range(10) for b in range(a + 1, 10)]
    ranks = np.argsort(np.argsort(-imps, axis=1), axis=1) + 1
    top10 = np.argsort(-imps.mean(0))[:10]
    res = dict(mean_pairwise_spearman=round(float(np.mean(rhos)), 3), min_pairwise_spearman=round(float(np.min(rhos)), 3),
               mean_rank_overall_top10=round(float(ranks[:, top10].mean()), 2),
               overall_top10=[feature_cols[i] for i in top10],
               fold_top5={k + 1: [feature_cols[i] for i in np.argsort(-imps[k])[:5]] for k in range(10)},
               top5_in_all_folds_top20=int(sum(all(ranks[k, i] <= 20 for k in range(10)) for i in top10[:5])))
    pd.DataFrame(imps.T, index=feature_cols, columns=[f"fold{k}" for k in range(1, 11)]).to_csv(os.path.join(CKPT_DIR, "shap_global_allfolds.csv"))
    print(json.dumps(res, indent=1)); R["shap_stability"] = res; _done("8g2")

# ---------- 8g-3 family ablation (6 families alone, clean run) ----------
if "8g3" in DONE and "8g3" not in RERUN:
    print("=== 8g-3 already completed — skipping")
else:
    print("=== 8g-3) FAMILY ABLATION, each family alone ===")
    LEADS = ("I_", "II_", "III_", "aVR_", "aVL_", "aVF_", "V1_", "V2_", "V3_", "V4_", "V5_", "V6_")
    # Explicit mapping that reproduces Table 2 exactly (182/168/145/27/15/6), fixed 2026-10-09.
    LEADS = ("I_", "II_", "III_", "aVR_", "aVL_", "aVF_", "V1_", "V2_", "V3_", "V4_", "V5_", "V6_")
    COMPLEX_T = {"Amplitude_Entropy", "Hjorth_Activity", "Hjorth_Complexity", "Hjorth_Mobility", "Kurtosis", "RMS", "Range",
                 "Shannon_Entropy", "Signal_Energy", "Skewness", "ZCR", "N_Valid_Beats", "PTFV1_N_Valid_Beats"}
    INTERVAL_T = {"PR_Interval", "PR_Segment", "P_Duration", "QRS_Duration", "QRS_Duration_Extended", "QRS_Offset_Rel_R",
                  "QRS_Offset_Rel_R_Extended", "QRS_Onset_Rel_R", "QT_Interval", "QTc_Bazett", "QTc_Framingham", "QTc_Fridericia",
                  "ST_Interval", "T_Wave_Duration"}
    MORPH_T = {"P_Amplitude", "Q_Amplitude", "R_Amplitude", "S_Amplitude", "T_Amplitude", "Q_R_Ratio", "RS_Ratio", "QRS_Net_Area",
               "ST80_Level", "ST_J_Level", "ST_Level", "ST_Slope", "T_Amplitude_Variability", "T_Wave_Slope", "P_Terminal_Neg_Depth_mV"}
    MORPH_G = {"Sokolow_Lyon_Index", "Frontal_QRS_Axis_Degrees", "Global_QRS_Vector_Mag"}
    def family_of(c):
        if c.startswith("HRV_") or c == "N_RPeaks": return "HRV"
        if "fQRS" in c: return "fQRS"
        base = c
        for l in sorted(LEADS, key=len, reverse=True):
            if c.startswith(l): base = c[len(l):]; break
        if base in COMPLEX_T: return "Complexity"
        if base in INTERVAL_T: return "Intervals"
        if base in MORPH_T or c in MORPH_G or c.startswith("Ratio_"): return "Morphology"
        return "Global"
    fam = pd.Series({c: family_of(c) for c in feature_cols})
    counts = fam.value_counts().to_dict(); print("auto family counts:", counts)
    EXPECTED = {"Morphology": 182, "Intervals": 168, "Complexity": 145, "fQRS": 27, "Global": 15, "HRV": 6}
    if counts != EXPECTED:
        print("!! auto-assignment does not reproduce Table 2 counts", EXPECTED)
        raise SystemExit("mapping mismatch, stop")
    fam.to_csv(os.path.join(CKPT_DIR, "feature_family_map.csv"), header=["family"])
    out = {}
    for f_name in ["Morphology", "Intervals", "Complexity", "Global", "fQRS", "HRV"]:
        keep = [c for c in feature_cols if fam[c] == f_name]; excl = [c for c in feature_cols if c not in keep]
        t0 = time.time(); p = run_cv(df, X, y, feature_cols, exclude=excl, tag=f"fam_{f_name}"); mm = metrics(y, p)
        out[f_name] = dict(n_features=len(keep), accuracy=round(float(mm["accuracy"]), 4), macro_f1=round(float(mm["macro_f1"]), 4), macro_auc=round(float(mm["macro_auc"]), 4))
        print(f"  {f_name:11s} ({len(keep):3d}) acc {mm['accuracy']:.4f} | mF1 {mm['macro_f1']:.4f} | mAUC {mm['macro_auc']:.4f}  [{(time.time()-t0)/60:.0f} min]")
        R["ablation_families"] = dict(mapping_matches_table2=(counts == EXPECTED), counts=counts, results=out)
        json.dump({"R": R, "DONE": sorted(DONE), "ts": time.strftime("%Y-%m-%d %H:%M:%S")}, open(os.path.join(CKPT_DIR, "state.json"), "w"), default=str)
    _done("8g3")
print("SECTION 8g DONE")


## 9) COPY-PASTE SUMMARY — send this whole block back

In [ ]:
try:
    _st = os.path.join(CKPT_DIR, "state.json")
except NameError:
    _st = "/content/drive/MyDrive/PTB-XL Dataset Analysis/_analysis_checkpoints/state.json"
if os.path.exists(_st):
    _s = json.load(open(_st))
    R = _s["R"]
    _miss = [s for s in ["1", "2", "3", "4", "5", "6", "7", "8", "8b", "8c", "8d", "8e"] if s not in _s["DONE"]]
    print(f"[checkpoint] summary state loaded — done: {sorted(_s['DONE'])}")
    if _miss:
        print("[checkpoint] note: not complete:", _miss, "(section 7 = multi-label evaluation; বাকি section গুলো আগে complete করুন)")
out = {}
out["notebook_version"] = "2.5"
out["baseline"] = {k: (round(v, 4) if isinstance(v, float) else v) for k, v in R.get("baseline", {}).items() if k != "error_breakdown"}
if "baseline" in R and "error_breakdown" in R.get("baseline", {}):
    out["error_breakdown"] = {k: v for k, v in R["baseline"]["error_breakdown"].items() if k != "confusion_matrix"}
out["permutation"] = R.get("permutation")
out["leave_out"] = R.get("leave_out")
out["iq"] = R.get("iq")
out["unweighted"] = R.get("unweighted")
out["ece"] = R.get("ece")
out["cinc"] = R.get("cinc", "skipped")
out["shap"] = R.get("shap")
out["cinc_fmax_ci"] = R.get("cinc_fmax_ci")
out["shap_stability"] = R.get("shap_stability")
out["ablation_families"] = R.get("ablation_families")
out["ablation_nohrv"] = R.get("ablation_nohrv")
out["unambiguous_subset"] = R.get("unambiguous_subset", "skipped")
out["figures"] = R.get("figures", "skipped")
out["ablation_nohrv"] = R.get("ablation_nohrv", "skipped")
out["perfold"] = R.get("perfold", "skipped")
with open("analysis_results.json", "w") as f:
    json.dump(out, f, indent=2, default=str)
print(json.dumps(out, indent=2, default=str))
print("\n=== END — send everything above (or the file analysis_results.json) back ===")